In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import math
import os
from datetime import datetime, timezone, timedelta

import pandas as pd


# 路径配置（按实际情况修改 BASE_DIR 即可）

BASE_DIR = r"D:\DA\Huawei_Sleep_Analysis\DATA"
RAW_DIR  = os.path.join(BASE_DIR, "raw")
OUT_DIR  = os.path.join(BASE_DIR, "processed")
os.makedirs(OUT_DIR, exist_ok=True)

HEART_FILE = os.path.join(RAW_DIR, "heart_health_detail.json")
SLEEP_FILE = os.path.join(RAW_DIR, "sleep_health_detail.json")
SPORT_FILE = os.path.join(RAW_DIR, "sport_health_detail.json")
OUTPUT_CSV  = os.path.join(OUT_DIR, "health_daily_summary.csv")

# 时区：UTC+8
TZ8 = timezone(timedelta(hours=8))

# 卡路里原始单位换算系数（1 kcal = 4184 J，华为存储单位为 mJ / 0.001 kcal）
CAL_DIVISOR = 4184.0



# 1. 解析睡眠数据

SLEEP_KEY_MAP = {
    "PROFESSIONAL_SLEEP_DREAM":   "REM时长",
    "PROFESSIONAL_SLEEP_SHALLOW": "浅睡时长",
    "PROFESSIONAL_SLEEP_DEEP":    "深睡时长",
    "PROFESSIONAL_SLEEP_WAKE":    "清醒时长",
    "PROFESSIONAL_SLEEP_NOON":    "午睡时长",
}

def parse_sleep(filepath: str) -> pd.DataFrame:
    """
    返回列：日期 | REM时长 | 午睡时长 | 浅睡时长 | 深睡时长 | 清醒时长
            (时长单位：分钟)
    以及衍生列：夜间总睡眠(h) | 在床时间(min) | 睡眠效率(%) |
               深睡比例(%) | 浅睡比例(%) | 快速眼动比例(%) | 是否午睡
    """
    with open(filepath, encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for rec in raw:
        for sp in rec.get("samplePoints", []):
            key = sp["key"]
            if key not in SLEEP_KEY_MAP:
                continue
            duration_min = (sp["endTime"] - sp["startTime"]) / 60_000

            if key == "PROFESSIONAL_SLEEP_NOON":
                # 午睡：按开始时间归属日期
                day = datetime.fromtimestamp(
                    sp["startTime"] / 1000, tz=TZ8
                ).strftime("%Y-%m-%d")
            else:
                # 夜间睡眠：按结束时间归属日期（深夜开始 → 次日早晨结束）
                day = datetime.fromtimestamp(
                    sp["endTime"] / 1000, tz=TZ8
                ).strftime("%Y-%m-%d")

            rows.append({
                "日期":  day,
                "字段":  SLEEP_KEY_MAP[key],
                "时长":  duration_min,
            })

    df = (
        pd.DataFrame(rows)
        .groupby(["日期", "字段"])["时长"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )

    # 确保所有列都存在（某些天可能缺某类型）
    for col in ["REM时长", "午睡时长", "浅睡时长", "深睡时长", "清醒时长"]:
        if col not in df.columns:
            df[col] = 0.0


    # 夜间总睡眠（小时）= (REM + 浅睡 + 深睡) / 60
    df["夜间总睡眠"] = ((df["REM时长"] + df["浅睡时长"] + df["深睡时长"]) / 60).round(2)

    # 在床时间（分钟）= REM + 浅睡 + 深睡 + 清醒
    df["在床时间"] = (df["REM时长"] + df["浅睡时长"] + df["深睡时长"] + df["清醒时长"]).round(2)

    # 睡眠效率（%）= 实际睡眠 / 在床时间 × 100
    df["睡眠效率"] = (
        (df["REM时长"] + df["浅睡时长"] + df["深睡时长"]) / df["在床时间"] * 100
    ).round(2)

    # 各阶段占实际睡眠比例（%）
    actual = df["REM时长"] + df["浅睡时长"] + df["深睡时长"]
    df["深睡比例"]    = (df["深睡时长"]  / actual * 100).round(2)
    df["浅睡比例"]    = (df["浅睡时长"]  / actual * 100).round(2)
    df["快速眼动比例"] = (df["REM时长"]   / actual * 100).round(2)

    # 是否午睡
    df["是否午睡"] = df["午睡时长"] > 0

    # 时长列取整（分钟保留两位小数，但原表为整数 → 做 round 后转 int 兼容）
    for col in ["REM时长", "午睡时长", "浅睡时长", "深睡时长", "清醒时长"]:
        df[col] = df[col].round(2)

    return df



# 2. 解析运动数据


def parse_sport(filepath: str) -> pd.DataFrame:
    """
    返回列：日期 | 总步数 | 总消耗(kcal) | 总时长(min) | 每公里消耗(kcal/km)
    """
    with open(filepath, encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for day_rec in raw:
        day_str = str(day_rec["recordDay"])  # e.g. 20250620
        day = f"{day_str[:4]}-{day_str[4:6]}-{day_str[6:]}"

        total_steps = 0
        total_cal   = 0   # 原始单位（mJ 级别）
        total_dur   = 0   # 分钟
        total_dist  = 0   # 米

        for entry in day_rec.get("sportDataUserData", []):
            for info in entry.get("sportBasicInfos", []):
                total_steps += info.get("steps",    0)
                total_cal   += info.get("calorie",  0)
                total_dur   += info.get("duration", 0)
                total_dist  += info.get("distance", 0)

        rows.append({
            "日期":    day,
            "总步数":   total_steps,
            "_cal_raw": total_cal,
            "总时长":   total_dur,
            "_dist_m":  total_dist,
        })

    df = pd.DataFrame(rows)

    # 卡路里换算：原始值 ÷ 4184 → kcal
    df["总消耗"] = (df["_cal_raw"] / CAL_DIVISOR).round(2)

    # 每公里消耗 = 总消耗(kcal) / 总距离(km)
    dist_km = df["_dist_m"] / 1000
    df["每公里消耗"] = (df["总消耗"] / dist_km.replace(0, float("nan"))).round(2)
    df["每公里消耗"] = df["每公里消耗"].fillna(0)

    return df[["日期", "总步数", "总消耗", "总时长", "每公里消耗"]]


# 3. 解析心率数据

def parse_heart(filepath: str) -> pd.DataFrame:
    """
    返回列：日期 | 平均心率 | 最小心率 | 最大心率 | 记录条数
    """
    with open(filepath, encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for rec in raw:
        for sp in rec.get("samplePoints", []):
            if sp["key"] != "DYNAMIC_HEART_RATE":
                continue
            try:
                bpm = json.loads(sp["value"])["bpm"]
            except (json.JSONDecodeError, KeyError):
                continue
            day = datetime.fromtimestamp(
                sp["startTime"] / 1000, tz=TZ8
            ).strftime("%Y-%m-%d")
            rows.append({"日期": day, "bpm": bpm})

    df_raw = pd.DataFrame(rows)
    df = (
        df_raw.groupby("日期")["bpm"]
        .agg(平均心率="mean", 最小心率="min", 最大心率="max", 记录条数="count")
        .reset_index()
    )
    df["平均心率"] = df["平均心率"].round(2)
    return df

# 4. 合并 & 导出


WEEKDAY_MAP = {0: "周一", 1: "周二", 2: "周三", 3: "周四",
               4: "周五", 5: "周六", 6: "周日"}

FINAL_COLS = [
    "日期", "REM时长", "午睡时长", "浅睡时长", "深睡时长", "清醒时长",
    "夜间总睡眠", "在床时间", "睡眠效率", "深睡比例", "浅睡比例", "快速眼动比例",
    "是否午睡", "星期", "总步数", "总消耗", "总时长", "每公里消耗",
    "平均心率", "最小心率", "最大心率", "记录条数",
]


def main():
    print("📂 读取原始数据…")
    df_sleep = parse_sleep(SLEEP_FILE)
    df_sport = parse_sport(SPORT_FILE)
    df_heart = parse_heart(HEART_FILE)

    print(f"  睡眠记录：{len(df_sleep)} 天")
    print(f"  运动记录：{len(df_sport)} 天")
    print(f"  心率记录：{len(df_heart)} 天")

    # 以心率数据的日期范围为主键（心率每天基本都有），outer join 保留全部日期
    all_dates = sorted(
        set(df_sleep["日期"].tolist())
        | set(df_sport["日期"].tolist())
        | set(df_heart["日期"].tolist())
    )
    df = pd.DataFrame({"日期": all_dates})

    df = df.merge(df_sleep, on="日期", how="left")
    df = df.merge(df_sport, on="日期", how="left")
    df = df.merge(df_heart, on="日期", how="left")

    # 星期
    df["星期"] = pd.to_datetime(df["日期"]).dt.weekday.map(WEEKDAY_MAP)

    # 列顺序
    df = df[FINAL_COLS]

    # 数值格式统一：小数保留两位（整数列不受影响）
    float_cols = [
        "夜间总睡眠", "在床时间", "睡眠效率",
        "深睡比例", "浅睡比例", "快速眼动比例",
        "总消耗", "每公里消耗", "平均心率",
    ]
    for col in float_cols:
        df[col] = df[col].round(2)

    # 睡眠时长列（分钟）：原表为整数，有数据时取 int，NaN 行保持 NaN
    sleep_min_cols = ["REM时长", "午睡时长", "浅睡时长", "深睡时长", "清醒时长"]
    for col in sleep_min_cols:
        df[col] = df[col].round(2)

    print(f"\n✅ 合并完成，共 {len(df)} 行")
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"📄 已导出至：{OUTPUT_CSV}")
    print("\n前 3 行预览：")
    print(df.head(3).to_string(index=False))


if __name__ == "__main__":
    main()